# f#μ and ν from Python

The interactive tool is the web app: run `scenic-dmp serve examples/pedestrian.scenic` and open
http://127.0.0.1:8765/ to drag ν, read off the patched program P′ and check it by sampling. This notebook drives the
same `Session` from code.

`summary()` shows the estimated distribution f#μ̂ of a view f with 95% CIs, the desired ν, the Jeffrey weights
ŵ(y) = ν(y) / f#μ̂(y), the acceptance probabilities â(y) = ŵ(y) / M, and the acceptance rate 1/M of the `require`
patch. `preview(g)` shows how a second view g changes under μ̂′, the cached samples reweighted by ŵ(f(sᵢ)).

A view is a list of Scenic expressions over the program's top-level names, a plain Python function of their
sampled values, and a declared finite set of values (or numeric bin edges). Views built with `View.from_spec`, as
the web app makes them, can also be written into P′ with `session_patch`.

In [1]:
import math
from pathlib import Path

from scenic_dmp import Program, Session, View, deg

EXAMPLES = Path("..") / "examples"
SCENIC = Path.home() / "research/scenic-example/vendor/Scenic"

/Users/abasu/research/scenic-dmp-tool/.venv/lib/python3.11/site-packages/scenic/core/errors.py:271: UserWarning: unable to install sys.excepthook to format Scenic backtraces
  warnings.warn("unable to install sys.excepthook to format Scenic backtraces")


## 1. The example from `example.md`

f(s) = brake if d ≤ 2 and the pedestrian is crossing (c = 90°), otherwise continue. The exact answer is
f#μ(brake) = 0.35. Scenic angles are in radians, so compare with `math.isclose(c, deg(90))`.

In [ ]:
P = Program.from_file(EXAMPLES / "pedestrian.scenic")

brake = View(
    "brake",
    ["d", "c"],
    lambda d, c: "brake" if d <= 2 and math.isclose(c, deg(90)) else "continue",
    values=["brake", "continue"],
)
d_view = View("d", ["d"], int, values=[1, 2, 3])
c_view = View("c", ["c"], lambda c: "crossing" if math.isclose(c, deg(90)) else "along", values=["crossing", "along"])

s = Session(P, brake, also=[d_view, c_view], seed=0).sample(5000)
print(s.summary())

Set **brake** to 50%. The readout should show ŵ ≈ 1.43 / 0.77, M ≈ 1.43 and acceptance 1/M ≈ 70%,
matching `example.md`. The preview of **d** shows d = 1 and d = 2 gaining mass.

In [3]:
s.set_nu({"brake": 0.5, "continue": 0.5})
print(s.summary())
s.preview("d")

view 'brake': n = 5000, P iterations/scene = 1
M = 1.439, acceptance 1/M = 0.6948, P' iterations/scene ≈ 1.44
  brake        1737/5000   34.74% [33.43%, 36.07%]  ν  50.00%  ŵ 1.439  â 1
  continue     3263/5000   65.26% [63.93%, 66.57%]  ν  50.00%  ŵ 0.7662  â 0.5323


{'name': 'd',
 'labels': ['1', '2', '3'],
 'before': array([0.2032, 0.4954, 0.3014]),
 'after': array([0.22380237, 0.54527517, 0.23092246]),
 'n_eff': np.float64(4534.264799999999)}

Changing a view's function or its values reclassifies the cached scenes without resampling. Only new argument
expressions need a resample. Here f gets a stricter threshold:

In [4]:
s.set_view(View(
    "brake",
    ["d", "c"],
    lambda d, c: "brake" if d <= 1 and math.isclose(c, deg(90)) else "continue",
    values=["brake", "continue"],
))
print(s.summary())

view 'brake': n = 5000, P iterations/scene = 1
M = 4.941, acceptance 1/M = 0.2024, P' iterations/scene ≈ 4.94
  brake         506/5000   10.12% [ 9.31%, 10.99%]  ν  50.00%  ŵ 4.941  â 1
  continue     4494/5000   89.88% [89.01%, 90.69%]  ν  50.00%  ŵ 0.5563  â 0.1126


### Values with no samples

A declared value that never occurs is shown greyed out, labelled "0 of n" and locked at ν = 0,
since reweighting can't create mass there.

In [ ]:
brake3 = View(
    "brake3",
    ["d", "c"],
    lambda d, c: "brake" if d <= 2 and math.isclose(c, deg(90)) else "continue",
    values=["brake", "continue", "swerve"],
)
print(Session(P, brake3, seed=0).sample(2000).summary())

## 2. A driving program: `badlyParkedCarPullingIn.scenic` (Town05)

The map comes from the vendored Scenic checkout. The program requires the parked car to be more than 20 m from ego,
so "< 20" has no samples. Notice how few samples "≥ 50" has: pushing it up makes M, and so the cost
of the patched program, explode. Sampling runs at about 40 scenes/s.

In [ ]:
driving = Program.from_file(SCENIC / "examples/driving/badlyParkedCarPullingIn.scenic", mode2D=True)
distance = View("distance", ["distance from ego to parkedCar"], float, bins=[20, 25, 30, 35, 40, 45, 50])
side = View("angle side", ["badAngle"], lambda a: "left" if a > 0 else "right", values=["left", "right"])

sd = Session(driving, distance, also=[side], seed=0).sample(500)
print(sd.summary())

## 3. The patched program P′

A view built from a spec (Scenic bindings plus a Python expression, as in the web app) can be written into P.
`session_patch` appends one `require not (f(s) == y) or u < â(y)` line per value with â(y) < 1, and `validate`
samples P′ to check that f#μ̂_{P′} ≈ ν and that P′ costs about M times P's iterations per scene.

In [ ]:
from scenic_dmp import session_patch, validate

brake_spec = View.from_spec({
    "name": "brake",
    "bindings": [{"name": "d", "expr": "d"}, {"name": "c", "expr": "c"}],
    "expr": '"brake" if d <= 2 and isclose(c, deg(90)) else "continue"',
    "values": ["brake", "continue"],
})
sp = Session(P, brake_spec, seed=0).sample(5000)
sp.set_nu({"brake": 0.5, "continue": 0.5})
print(session_patch(sp)["block"])

In [ ]:
result, _ = validate(sp, 3000)
print(f"f#μ̂_P′ = {dict(zip(result['labels'], result['p_hat']))}, ν inside the 95% CIs: {result['nu_in_ci']}")
print(f"P′ iterations/scene: {result['mean_iters']:.3g} (predicted {result['predicted_iters']:.3g})")